# Online Autocorrelation

Autocovariances and autocorrelations of a stream with `AutoCov` from [OnlineStats.jl](https://github.com/joshday/OnlineStats.jl), a correlogram, a time window over time-stamped observations, and a dependence that changes.

In [ ]:
Pollis.packages("OnlineStats", "Plots")

In [ ]:
using OnlineStats, Random, Statistics, Dates, Plots

Random.seed!(1)  # simulated: an AR(1) stream, y(t) = 0.7 y(t-1) + e(t), whose autocorrelation at lag k is 0.7^k
e = randn(10_000); data = zeros(10_000); data[1] = e[1]
for t in 2:10_000; data[t] = 0.7 * data[t - 1] + e[t]; end  # Replace data with your own stream, in time order.

## Autocorrelations up to Lag 10

`AutoCov(10)` keeps the last ten observations and running averages of the products with each of them.

In [ ]:
o = fit!(AutoCov(10), data)
println("Online autocorrelations: ", round.(autocor(o), digits=3))
println("True autocorrelations:   ", round.(0.7 .^ (0:10), digits=3))

## A Correlogram

The bars are the online estimates; the dots are the true values 0.7^k.

In [ ]:
bar(0:10, autocor(o), xlabel="Lag", ylabel="Autocorrelation", label="AutoCov")
scatter!(0:10, 0.7 .^ (0:10), label="0.7^k")

## A Time Window Over Time-Stamped Observations

Here the observations arrive at irregular intervals, 1 to 5 seconds apart. `MovingTimeWindow` keeps those within 30 minutes of the latest time stamp, counted from the data, not from the clock, so the number it holds varies.

In [ ]:
times = DateTime(2026, 1, 2, 9, 30) .+ Second.(cumsum(rand(1:5, length(data))))  # irregular arrivals
window = fit!(MovingTimeWindow(Minute(30); timetype=DateTime, valtype=Float64), zip(times, data))
kept = last.(value(window))  # the values in the window, in time order
println("Observations in the last 30 minutes: ", length(kept))
println("Autocorrelations in the window:      ", round.(autocor(fit!(AutoCov(5), kept)), digits=3))

## A Dependence That Changes

For the first half the coefficient is 0.7, for the second half -0.5, so the lag-1 autocorrelation changes sign. The incremental estimate averages the two halves, while the exponentially weighted one, span 500, follows the change.

In [ ]:
phi = vcat(fill(0.7, 5_000), fill(-0.5, 5_000))
z = randn(10_000); y = zeros(10_000); y[1] = z[1]
for t in 2:10_000; y[t] = phi[t] * y[t - 1] + z[t]; end

equal = AutoCov(1)
decaying = AutoCov(1, weight=ExponentialWeight(500))
r1e = Float64[]; r1w = Float64[]
for x in y
    fit!(equal, x); fit!(decaying, x)
    push!(r1e, nobs(equal) > 2 ? autocor(equal)[2] : NaN)
    push!(r1w, nobs(decaying) > 2 ? autocor(decaying)[2] : NaN)
end
plot([r1e r1w], label=["Incremental" "Exponentially Weighted"], xlabel="Observations", ylabel="Lag-1 autocorrelation")
plot!(phi, linestyle=:dash, label="True lag-1 autocorrelation")